# Create Customer ReAct Agent for YT videos summarizing

In [39]:
import re
from pytube import YouTube
from langchain_core.tools import tool
from IPython.display import display, JSON, Markdown
import yt_dlp
from typing import List, Dict
from langchain_core.messages import HumanMessage
from langchain_core.messages import ToolMessage
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_openai import ChatOpenAI


from youtube_transcript_api import YouTubeTranscriptApi
from pytube import Search

import json

from dotenv import load_dotenv
load_dotenv()

# Suppress warnings
import warnings
warnings.filterwarnings("ignore")

# Suppress pytube errors
import logging
pytube_logger = logging.getLogger('pytube')
pytube_logger.setLevel(logging.ERROR)

# Suppress yt-dlp warnings
yt_dlp_logger = logging.getLogger('yt_dlp')
yt_dlp_logger.setLevel(logging.ERROR)

In [3]:
llm = ChatOpenAI(model="gpt-4o-mini")

## Tool list

In [4]:
tools = []

## Tools Creation

### Defining video ID extraction tool

In [5]:
@tool
def extract_video_id(url: str) -> str:
    """
    Extracts the 11-character YouTube video ID from a URL.
    
    Args:
        url (str): A YouTube URL containing a video ID.

    Returns:
        str: Extracted video ID or error message if parsing fails.
    """
    
    # Regex pattern to match video IDs
    pattern = r'(?:v=|be/|embed/)([a-zA-Z0-9_-]{11})'
    match = re.search(pattern, url)
    return match.group(1) if match else "Error: Invalid YouTube URL"

In [7]:
extract_video_id.run("https://www.youtube.com/watch?v=hfIUstzHs9A")

'hfIUstzHs9A'

In [7]:
tools.append(extract_video_id)

### Defining transcript fetching tool

In [6]:
@tool
def fetch_transcript(video_id: str, language: str = "en") -> str:
    """
    Fetches the transcript of a YouTube video.
    
    Args:
        video_id (str): The YouTube video ID (e.g., "dQw4w9WgXcQ").
        language (str): Language code for the transcript (e.g., "en", "es").
    
    Returns:
        str: The transcript text or an error message.
    """
    
    try:
        ytt_api = YouTubeTranscriptApi()
        transcript = ytt_api.fetch(video_id, languages=[language])
        print(transcript)
        return " ".join([snippet.text for snippet in transcript.snippets])
    except Exception as e:
        return f"Error: {str(e)}"

In [13]:
fetch_transcript.invoke({"video_id": "hfIUstzHs9A"})

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='Over the past couple of months, large language models, or LLMs, such as chatGPT, have taken the world by storm.', start=0.18, duration=7.8), FetchedTranscriptSnippet(text="Whether it's writing poetry or helping plan your upcoming vacation, we are seeing a step change in the performance of AI and its potential to drive enterprise value.", start=8.79, duration=10.38), FetchedTranscriptSnippet(text='My name is Kate Soule.', start=19.98, duration=1.0), FetchedTranscriptSnippet(text="I'm a senior manager of business strategy at IBM Research,", start=21.1, duration=3.258), FetchedTranscriptSnippet(text="and today I'm going to give a brief overview of this new field of AI that's emerging and how it can be used in a business setting to drive value.", start=24.358, duration=7.681), FetchedTranscriptSnippet(text='Now, large language models are actually a part of a different class of models called foundation models.', start=32.99, duratio

'Over the past couple of months, large language models, or LLMs, such as chatGPT, have taken the world by storm. Whether it\'s writing poetry or helping plan your upcoming vacation, we are seeing a step change in the performance of AI and its potential to drive enterprise value. My name is Kate Soule. I\'m a senior manager of business strategy at IBM Research, and today I\'m going to give a brief overview of this new field of AI that\'s emerging and how it can be used in a business setting to drive value. Now, large language models are actually a part of a different class of models called foundation models. Now, the term "foundation models" was actually first coined by a team from Stanford when they saw that the field of AI was converging to a new paradigm. Where before AI applications were being built by training, maybe a library of different AI models, where each AI model was trained on very task-specific data to perform very specific task. They predicted that we were going to start 

In [8]:
tools.append(fetch_transcript)

### Defining YouTube search tool

In [9]:
@tool
def search_youtube(query: str) -> List[Dict[str, str]]:
    """
    Search YouTube for videos matching the query.
    
    Args:
        query (str): The search term to look for on YouTube
        
    Returns:
        List of dictionaries containing video titles and IDs in format:
        [{'title': 'Video Title', 'video_id': 'abc123'}, ...]
        Returns error message if search fails
    """
    try:
        s = Search(query)
        return [
            {
                "title": yt.title,
                "video_id": yt.video_id,
                "url": f"https://youtu.be/{yt.video_id}"
            }
            for yt in s.results
        ]
    except Exception as e:
        return f"Error: {str(e)}"

In [17]:
search_out=search_youtube.run("Generative AI")
display(JSON(search_out))

<IPython.core.display.JSON object>

In [19]:
search_out[0]

{'title': 'Generative AI Explained In 5 Minutes | What Is GenAI? | Introduction To Generative AI | Simplilearn',
 'video_id': 'NRmAXDWJVnU',
 'url': 'https://youtu.be/NRmAXDWJVnU'}

In [11]:
tools.append(search_youtube)

### Defining metadata extraction tool

In [12]:
@tool
def get_full_metadata(url: str) -> dict:
    """Extract metadata given a YouTube URL, including title, views, duration, channel, likes, comments, and chapters."""
    with yt_dlp.YoutubeDL({'quiet': True, 'logger': yt_dlp_logger}) as ydl:
        info = ydl.extract_info(url, download=False)
        return {
            'title': info.get('title'),
            'views': info.get('view_count'),
            'duration': info.get('duration'),
            'channel': info.get('uploader'),
            'likes': info.get('like_count'),
            'comments': info.get('comment_count'),
            'chapters': info.get('chapters', [])
        }

In [22]:
meta_data=get_full_metadata.run("https://www.youtube.com/watch?v=T-D1OfcDW1M")
meta_data

{'title': 'What is Retrieval-Augmented Generation (RAG)?',
 'views': 1685668,
 'duration': 395,
 'channel': 'IBM Technology',
 'likes': 39925,
 'comments': 911,
 'chapters': [{'start_time': 0.0, 'title': 'Introduction', 'end_time': 18.0},
  {'start_time': 18.0, 'title': 'What is RAG', 'end_time': 42.0},
  {'start_time': 42.0, 'title': 'An anecdote', 'end_time': 82.0},
  {'start_time': 82.0, 'title': 'Two problems', 'end_time': 138.0},
  {'start_time': 138.0, 'title': 'Large language models', 'end_time': 263.0},
  {'start_time': 263.0, 'title': 'How does RAG help', 'end_time': 395}]}

In [13]:
tools.append(get_full_metadata)

### Defining thumbnail retrieval tool

In [14]:
@tool
def get_thumbnails(url: str) -> List[Dict]:
    """
    Get available thumbnails for a YouTube video using its URL.
    
    Args:
        url (str): YouTube video URL (any format)
        
    Returns:
        List of dictionaries with thumbnail URLs and resolutions in YouTube's native order
    """
    
    try:
        with yt_dlp.YoutubeDL({'quiet': True, 'logger': yt_dlp_logger}) as ydl:
            info = ydl.extract_info(url, download=False)
            
            thumbnails = []
            for t in info.get('thumbnails', []):
                if 'url' in t:
                    thumbnails.append({
                        "url": t['url'],
                        "width": t.get('width'),
                        "height": t.get('height'),
                        "resolution": f"{t.get('width', '')}x{t.get('height', '')}".strip('x')
                    })
            
            return thumbnails

    except Exception as e:
        return [{"error": f"Failed to get thumbnails: {str(e)}"}]

In [27]:
thumbnails=get_thumbnails.run("https://www.youtube.com/watch?v=qWHaMrR5WHQ")
thumbnails[0]

{'url': 'https://i.ytimg.com/vi/qWHaMrR5WHQ/3.jpg',
 'width': None,
 'height': None,
 'resolution': ''}

In [15]:
tools.append(get_thumbnails)

### Creating a tool mapping dictionary

In [16]:
tool_mapping = {
    "get_thumbnails" : get_thumbnails,
    "extract_video_id": extract_video_id,
    "fetch_transcript": fetch_transcript,
    "search_youtube": search_youtube,
    "get_full_metadata": get_full_metadata
}

## Binding tools with LLM

In [17]:
llm_with_tools = llm.bind_tools(tools)

## Tool calling process

In [ ]:
# Define the processing steps
def execute_tool(tool_call):
    """Execute single tool call and return ToolMessage"""
    try:
        result = tool_mapping[tool_call["name"]].invoke(tool_call["args"])
        return ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"]
        )
    except Exception as e:
        return ToolMessage(
            content=f"Error: {str(e)}",
            tool_call_id=tool_call["id"]
        )

## Building Custom summarization

In [42]:
query = "I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english"
print(query)

I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english


#### Step 1. Convert the input prompt to a HumanMessage

Wrapping the query string in a HumanMessage object, which is the standard way to format user inputs in LangChain

In [43]:
messages = [HumanMessage(content = query)]
print(messages)

[HumanMessage(content='I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english')]


#### Step 2. Pass the message to LLM with tools



In [45]:
response_1 = llm_with_tools.invoke(messages)
response_1

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_UjqSLXRTXG939g4fzUn4yx92', 'function': {'arguments': '{"url":"https://www.youtube.com/watch?v=T-D1OfcDW1M"}', 'name': 'extract_video_id'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 380, 'total_tokens': 409, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-631a9634-8367-4d5a-a7fd-e3ab41c29d6e-0', tool_calls=[{'name': 'extract_video_id', 'args': {'url': 'https://www.youtube.com/watch?v=T-D1OfcDW1M'}, 'id': 'call_UjqSLXRTXG939g4fzUn4yx92', 'type': 'tool_call'}], usage_metadata={'input_tokens': 380, 'output_tokens': 29, 'total_tokens': 409})

In [46]:
messages.append(response_1) # Add llm response to message history

#### Step 3. Extract tool calls from LLM response

In [48]:
tool_calls_1 = response_1.tool_calls
tool_calls_1

[{'name': 'extract_video_id',
  'args': {'url': 'https://www.youtube.com/watch?v=T-D1OfcDW1M'},
  'id': 'call_UjqSLXRTXG939g4fzUn4yx92',
  'type': 'tool_call'}]

In [51]:
tool_name=tool_calls_1[0]['name']
args=tool_calls_1[0]['args']

my_tool = tool_mapping[tool_name]

video_id =my_tool.invoke(args)
video_id

'T-D1OfcDW1M'

#### Step 4. Update message history with tool results

In [52]:
messages.append(ToolMessage(content = video_id, tool_call_id = tool_calls_1[0]['id']))

#### Step 5. Send updated messages back to LLM

In [53]:
response_2 = llm_with_tools.invoke(messages)
response_2

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_iXH2foE8BF6uAGOK9PtdIB4o', 'function': {'arguments': '{"video_id":"T-D1OfcDW1M","language":"en"}', 'name': 'fetch_transcript'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 426, 'total_tokens': 453, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-f63f5ff5-de8c-4970-9503-b56d1b2a3dde-0', tool_calls=[{'name': 'fetch_transcript', 'args': {'video_id': 'T-D1OfcDW1M', 'language': 'en'}, 'id': 'call_iXH2foE8BF6uAGOK9PtdIB4o', 'type': 'tool_call'}], usage_metadata={'input_tokens': 426, 'output_tokens': 27, 'total_tokens': 453})

In [54]:
messages.append(response_2)

#### Step 6. Repeat Step 3 Here:

In [55]:
tool_calls_2 = response_2.tool_calls
tool_calls_2

[{'name': 'fetch_transcript',
  'args': {'video_id': 'T-D1OfcDW1M', 'language': 'en'},
  'id': 'call_iXH2foE8BF6uAGOK9PtdIB4o',
  'type': 'tool_call'}]

In [56]:
fetch_transcript_tool_output = tool_mapping[tool_calls_2[0]['name']].invoke(tool_calls_2[0]['args'])
fetch_transcript_tool_output

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='Large language models. They are everywhere.', start=0.06, duration=2.572), FetchedTranscriptSnippet(text='They get some things amazingly right', start=2.632, duration=2.635), FetchedTranscriptSnippet(text='and other things very interestingly wrong.', start=5.267, duration=2.474), FetchedTranscriptSnippet(text='My name\xa0is Marina Danilevsky.', start=7.819, duration=1.759), FetchedTranscriptSnippet(text='I am a Senior Research Scientist here at IBM Research.', start=9.578, duration=2.658), FetchedTranscriptSnippet(text='And I want\xa0to tell you about a framework to help large language models', start=12.314, duration=4.235), FetchedTranscriptSnippet(text='be more accurate and more up to\xa0date:', start=16.549, duration=2.099), FetchedTranscriptSnippet(text='Retrieval-Augmented Generation, or RAG.', start=18.648, duration=3.252), FetchedTranscriptSnippet(text='Let\'s just talk about the "Generation" part for a\xa0minute.', star

'Large language models. They are everywhere. They get some things amazingly right and other things very interestingly wrong. My name\xa0is Marina Danilevsky. I am a Senior Research Scientist here at IBM Research. And I want\xa0to tell you about a framework to help large language models be more accurate and more up to\xa0date: Retrieval-Augmented Generation, or RAG. Let\'s just talk about the "Generation" part for a\xa0minute. So forget the "Retrieval-Augmented". So the\xa0generation, this refers to large language models,\xa0or LLMs, that generate text in response to a user query, referred to as a prompt. These\xa0models can have some undesirable behavior. I want to tell you an anecdote to illustrate this. So my kids, they recently asked me this question: "In our solar system, what planet has the most\xa0moons?" And my response was, “Oh, that\'s really great that you\'re asking this question. I loved\xa0space when I was your age.” Of course, that was like 30 years ago. But I know this! 

#### Step 6. Repeat Step 4 Here:

In [57]:
messages.append(ToolMessage(content = fetch_transcript_tool_output, tool_call_id = tool_calls_2[0]['id']))

#### Step 6. Repeat Step 5

In [58]:
summary = llm_with_tools.invoke(messages)

In [59]:
summary

AIMessage(content='The video features Marina Danilevsky, a Senior Research Scientist at IBM Research, discussing the concept of Retrieval-Augmented Generation (RAG) in large language models (LLMs). Here’s a summary of the key points:\n\n1. **Introduction to LLMs**: Danilevsky highlights the prevalence of large language models and their tendency to produce both accurate and inaccurate responses.\n\n2. **Challenges with LLMs**: She shares an anecdote about answering a question regarding which planet has the most moons. This illustrates two common issues with LLMs:\n   - Lack of sourcing for the information provided.\n   - Responses that may be outdated or incorrect.\n\n3. **Retrieval-Augmented Generation (RAG)**: Danilevsky introduces RAG as a framework designed to improve the accuracy and relevance of LLM responses by integrating a retrieval mechanism:\n   - Instead of solely relying on the model\'s training data, RAG allows the model to query a content store (which can be open like the

#### Step 7. Finally, extract just the content from the final message using RunnableLambda

In [60]:
summary.content

'The video features Marina Danilevsky, a Senior Research Scientist at IBM Research, discussing the concept of Retrieval-Augmented Generation (RAG) in large language models (LLMs). Here’s a summary of the key points:\n\n1. **Introduction to LLMs**: Danilevsky highlights the prevalence of large language models and their tendency to produce both accurate and inaccurate responses.\n\n2. **Challenges with LLMs**: She shares an anecdote about answering a question regarding which planet has the most moons. This illustrates two common issues with LLMs:\n   - Lack of sourcing for the information provided.\n   - Responses that may be outdated or incorrect.\n\n3. **Retrieval-Augmented Generation (RAG)**: Danilevsky introduces RAG as a framework designed to improve the accuracy and relevance of LLM responses by integrating a retrieval mechanism:\n   - Instead of solely relying on the model\'s training data, RAG allows the model to query a content store (which can be open like the internet or close

## Building summarization chain using Runnables

The workflow follows these steps:
1. Convert the input prompt to a HumanMessage
2. Pass the message to LLM with tools
3. Extract tool calls from LLM response
4. Update message history with tool results
5. Send updated messages back to LLM
6. Repeat steps 3-5 as needed
7. Finally, extract just the content from the final message using RunnableLambda

Each step maintains state using RunnablePassthrough until you reach the final message, at which point you'll apply RunnableLambda to extract only the summary text.

---

Our chain will only cater 2 tools calls. llm_response | tool_call_response | llm_response | tool_call_response | llm_response.\
If a query requires more than 2-step tool calls, then this flow will not cover it. Even, if a query will require only one tool call, our chain will still pass from all steps.

**Our query for this chain:**
`I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english`\

Steps for this query:
- Pass query to llm
- llm respond with tool_to_be_called info like: name (extract_video_id), arguments etc
- call tool (extract_video_id) with specified arguments and get response(video_id)
- Update message history with initial_user_query + AI_Message_of_tool_to_be_called + tool_response
- Pass updated message_history to llm
- llm respond with tool_to_be_called info like: name (fetch_transcript), arguments etc
- call tool (fetch_transcript) with specified arguments and get response(video_id)
- Update message history with previous_message_history + 2nd_AI_Message_of_tool_to_be_called + 2nd_tool_response
- Pass updated message_history to llm and get response(summary of youtube video)


### Creating the initial message setup

The first step of our chain that will handle the initial user query. The `RunnablePassthrough.assign` creates a component that takes an input dictionary containing a "query" and converts it into a list containing a single `HumanMessage` object.

In [19]:
# Start with initial query
initial_setup = RunnablePassthrough.assign(
    messages=lambda x: [HumanMessage(content=x["query"])]
)

### Defining the first LLM interaction

The second step of your chain will handle the first interaction with the language model. This component takes the formatted messages from the previous step, sends them to your tool-equipped LLM, and captures the response in a field called "ai_response."

In [20]:
# First LLM call (extract video ID)
first_llm_call = RunnablePassthrough.assign(
    ai_response=lambda x: llm_with_tools.invoke(x["messages"])
)

### Processing the first tool call

Here, we're defining the processing step that handles the LLM's first tool call. This component:
1. Executes each tool call by passing it to your `execute_tool` function, which runs the appropriate tool and returns the result as a `ToolMessage`
2. Updates the message history by combining the original messages, the LLM's response, with the tool calls, and the tool results
3. Prepares the updated conversation state for the next interaction with the LLM

In [21]:
# Process first tool call
first_tool_processing = RunnablePassthrough.assign(
    tool_messages=lambda x: [
        execute_tool(tc) for tc in x["ai_response"].tool_calls
    ]
).assign(
    messages=lambda x: x["messages"] + [x["ai_response"]] + x["tool_messages"] # Update message history
)

### Defining the second LLM interaction

Here, we're creating the next step in our chain that handles the second interaction with the language model. This component takes the updated message history (which now includes the results from the first tool call) and sends it to the LLM again.

In [22]:
# Second LLM call (fetch transcript)
second_llm_call = RunnablePassthrough.assign(
    ai_response2=lambda x: llm_with_tools.invoke(x["messages"])
)

### Processing the second tool call

Here, we're defining the processing step that handles the LLM's second tool call. Similar to the first tool processing step, this component executes the tool calls (typically fetching the transcript), creates tool messages with the results, and updates the message history by combining everything for the final summarization step.


In [23]:
# Process second tool call
second_tool_processing = RunnablePassthrough.assign(
    tool_messages2=lambda x: [
        execute_tool(tc) for tc in x["ai_response2"].tool_calls
    ]
).assign(
    messages=lambda x: x["messages"] + [x["ai_response2"]] + x["tool_messages2"] # Final message update
)

#### Generating the final summary

Here, you're defining the final step that produces the summary of the YouTube video. This component:
1. Takes the complete message history (which now contains the original query, tool calls, and tool results)
2. Invokes the LLM one last time to generate a summary
3. Extracts just the content field from the LLM's response
4. Uses a RunnableLambda to return only the summary text as the final output


In [24]:
# Generate final summary
final_summary = RunnablePassthrough.assign(
    summary=lambda x: llm_with_tools.invoke(x["messages"]).content
) | RunnableLambda(lambda x: x["summary"]) # Return just the summary text

In [25]:
chain = (
    initial_setup
    | first_llm_call
    | first_tool_processing
    | second_llm_call
    | second_tool_processing
    | final_summary
)

In [46]:
query = {"query": "I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english"}
result = chain.invoke(query)
print("Video Summary:\n", result)

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='Large language models. They are everywhere.', start=0.06, duration=2.572), FetchedTranscriptSnippet(text='They get some things amazingly right', start=2.632, duration=2.635), FetchedTranscriptSnippet(text='and other things very interestingly wrong.', start=5.267, duration=2.474), FetchedTranscriptSnippet(text='My name\xa0is Marina Danilevsky.', start=7.819, duration=1.759), FetchedTranscriptSnippet(text='I am a Senior Research Scientist here at IBM Research.', start=9.578, duration=2.658), FetchedTranscriptSnippet(text='And I want\xa0to tell you about a framework to help large language models', start=12.314, duration=4.235), FetchedTranscriptSnippet(text='be more accurate and more up to\xa0date:', start=16.549, duration=2.099), FetchedTranscriptSnippet(text='Retrieval-Augmented Generation, or RAG.', start=18.648, duration=3.252), FetchedTranscriptSnippet(text='Let\'s just talk about the "Generation" part for a\xa0minute.', star

#### Testing the Chain with a Different Query

In [26]:
query = {"query": "Get top 3 youtube videos of AgenticAI and their metadata"}
try:
    result = chain.invoke(query)
    print("Video Summary:\n", result)
except Exception as e:
    print("Non-critical network error:", e)

Video Summary:
 Here are the top 3 YouTube videos related to "AgenticAI" along with their metadata:

1. **Title:** [What is Agentic AI and How Does it Work?](https://youtu.be/15_pppse4fY)
   - **Channel:** codebasics
   - **Views:** 544,934
   - **Duration:** 13 minutes 49 seconds
   - **Likes:** 7,465
   - **Comments:** 312
   - **Chapters:**
     - Introduction (0:00 - 0:13)
     - RAG Based AI System (0:13 - 1:06)
     - Tool Augmented AI System (1:06 - 2:08)
     - Agentic AI System (2:08 - 8:02)
     - Agentic AI Apps with Code (8:02 - 9:31)
     - Low Code Agentic AI (9:31 - 11:16)
     - AI Agent vs Agentic AI (11:16 - 11:42)
     - Gen AI vs Agentic AI (11:42 - 13:49)

2. **Title:** [Generative vs Agentic AI: Shaping the Future of AI Collaboration](https://youtu.be/EDb37y_MhRw)
   - **Channel:** IBM Technology
   - **Views:** 930,794
   - **Duration:** 7 minutes 18 seconds
   - **Likes:** 12,609
   - **Comments:** 412
   - **Chapters:**
     - Intro (0:00 - 0:15)
     - Generat

In [ ]:
result

## Recursive chain flow

Now that you've created a chain that works well for your specific two-step tool calling process, you need to consider more complex scenarios. Your current chain is limited to exactly two tool calls in a fixed sequence. In real-world applications, you might need a variable number of tool calls depending on the user's query - for example searching for videos on a topic and then getting transcripts for multiple results.

To handle these more complex scenarios, you'll build a recursive chain that can dynamically decide how many tool calls are needed and continue processing until all necessary information has been gathered.


In [ ]:
def execute_tool(tool_call):
    """Execute single tool call and return ToolMessage"""
    try:
        result = tool_mapping[tool_call["name"]].invoke(tool_call["args"])
        content = json.dumps(result) if isinstance(result, (dict, list)) else str(result) # json.dumps --> stringify JSON
    except Exception as e:
        content = f"Error: {str(e)}"
    
    return ToolMessage(
        content=content,
        tool_call_id=tool_call["id"]
    )

#### Defining the core processing logic

This function handles the core processing logic of your recursive chain. It takes the current conversation history and:

1. Identifies the most recent message in the conversation
2. Extracts all tool calls from that message and executes them in parallel using your `execute_tool` helper
3. Updates the message history by adding the tool response messages
4. Gets the next response from the language model based on the updated conversation
5. Returns the complete updated message history with both tool responses and the new LLM response


In [29]:
def process_tool_calls(messages):
    """Recursive tool call processor"""
    last_message = messages[-1]
    
    # Execute all tool calls in parallel
    tool_messages = [
        execute_tool(tc) 
        for tc in getattr(last_message, 'tool_calls', [])
    ]
    
    # Add tool responses to message history
    updated_messages = messages + tool_messages
    
    # Get next LLM response
    next_ai_response = llm_with_tools.invoke(updated_messages)
    
    return updated_messages + [next_ai_response]

#### Creating the recursive stopping condition

This function determines whether your recursive process should continue or terminate. It:

1. Takes the current message history and examines the last message
2. Checks if that message contains any tool calls using the `getattr` function (which safely handles cases where the attribute might not exist)
3. Returns a boolean value - `True` if there are more tool calls to process, and `False` when you reach a point where the LLM has provided a final answer without requesting additional tools


In [28]:
def should_continue(messages):
    """Check if you need another iteration"""
    last_message = messages[-1]
    return bool(getattr(last_message, 'tool_calls', None))


#### Implementing the recursive function

This function implements the actual recursion that powers your dynamic tool calling process:

1. It first checks the stopping condition using the `should_continue` function to determine if more tools need to be called
2. If more tool calls are needed, it processes those calls using your `process_tool_calls` function and then recursively calls itself with the updated messages
3. If no more tool calls are needed, it returns the final message history, which contains the complete conversation, including the LLM's final response

After defining this recursive function, you'll wrap it in a `RunnableLambda` to make it compatible with LangChain's chain architecture.


In [32]:
def _recursive_chain(messages):
    """Recursively process tool calls until completion"""
    if should_continue(messages):
        new_messages = process_tool_calls(messages)
        return _recursive_chain(new_messages)
    return messages

recursive_chain = RunnableLambda(_recursive_chain)

#### Building the complete universal chain

Now, you're assembling your final universal chain that can handle any type of query requiring any number of tool calls. This chain consists of three main steps:

1. The first step converts the user query into a properly formatted `HumanMessage` object
2. The second step sends this initial message to your tool-equipped LLM and adds the LLM's first response to the message history
3. The final step passes the conversation to your recursive chain, which will handle all subsequent tool calls until the LLM provides a final answer

This universal chain is much more flexible than your earlier fixed-step chain, as it can dynamically adapt to queries that require different numbers and types of tool calls.


In [33]:
universal_chain = (
    RunnableLambda(lambda x: [HumanMessage(content=x["query"])])
    | RunnableLambda(lambda messages: messages + [llm_with_tools.invoke(messages)])
    | recursive_chain
)

In [34]:
query_us = {"query": "Show top 3 US trending videos with metadata and thumbnails"}

try:
    response = universal_chain.invoke(query_us)
    print("\nUS Trending Videos:\n", response[-1])
except Exception as e:
    print("Non-critical network error while fetching US trending videos:", e)


US Trending Videos:
 content='Here are the top 3 trending videos in the US along with their metadata and thumbnails:\n\n### 1. **New trend unlocked 🔓 #trending #subscribe #newtrend**\n- **Channel:** life with Faha\n- **Views:** 3,393,447\n- **Duration:** 7 seconds\n- **Likes:** Not available\n- **Comments:** 396\n- **[Watch Video](https://youtu.be/jgsv66FstEM)**\n  \n![Thumbnail](https://i.ytimg.com/vi/jgsv66FstEM/hqdefault.jpg)\n\n---\n\n### 2. **Top Hits 2026 Playlist 🎧 Spotify 2026 Mix Playlist ✨ Trending Pop Hits 2026  ~ Best New Pop Songs**\n- **Channel:** Dynamic Deep House\n- **Views:** 12,635\n- **Duration:** 2 hours, 19 minutes, 50 seconds\n- **Likes:** 94\n- **Comments:** 2\n- **[Watch Video](https://youtu.be/iGzqzmLagFg)**\n\n![Thumbnail](https://i.ytimg.com/vi/iGzqzmLagFg/hqdefault.jpg)\n\n---\n\n### 3. **New fashion trick unlocked..😂 #shorts #trending #relatable #reels #fyp #viral #shortfeed #fashion**\n- **Channel:** Armink World 💜\n- **Views:** 119,874,286\n- **Duration

In [35]:
response

[HumanMessage(content='Show top 3 US trending videos with metadata and thumbnails'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_ikThmF4ExDpLOv7wEESeToo3', 'function': {'arguments': '{"query":"trending"}', 'name': 'search_youtube'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 367, 'total_tokens': 383, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-1217bb87-9c50-4262-9dd2-f511e09367d6-0', tool_calls=[{'name': 'search_youtube', 'args': {'query': 'trending'}, 'id': 'call_ikThmF4ExDpLOv7wEESeToo3', 'type': 'tool_call'}], usage_metadata={'input_tokens': 367, 'output_tokens': 16, 'total_tokens': 383}),
 ToolMessage(conte

In [40]:
display(Markdown(response[-1].content))

Here are the top 3 trending videos in the US along with their metadata and thumbnails:

### 1. **New trend unlocked 🔓 #trending #subscribe #newtrend**
- **Channel:** life with Faha
- **Views:** 3,393,447
- **Duration:** 7 seconds
- **Likes:** Not available
- **Comments:** 396
- **[Watch Video](https://youtu.be/jgsv66FstEM)**
  
![Thumbnail](https://i.ytimg.com/vi/jgsv66FstEM/hqdefault.jpg)

---

### 2. **Top Hits 2026 Playlist 🎧 Spotify 2026 Mix Playlist ✨ Trending Pop Hits 2026  ~ Best New Pop Songs**
- **Channel:** Dynamic Deep House
- **Views:** 12,635
- **Duration:** 2 hours, 19 minutes, 50 seconds
- **Likes:** 94
- **Comments:** 2
- **[Watch Video](https://youtu.be/iGzqzmLagFg)**

![Thumbnail](https://i.ytimg.com/vi/iGzqzmLagFg/hqdefault.jpg)

---

### 3. **New fashion trick unlocked..😂 #shorts #trending #relatable #reels #fyp #viral #shortfeed #fashion**
- **Channel:** Armink World 💜
- **Views:** 119,874,286
- **Duration:** 8 seconds
- **Likes:** 613,907
- **Comments:** 1,700
- **[Watch Video](https://youtu.be/MO4OKdsh0Wo)**

![Thumbnail](https://i.ytimg.com/vi/MO4OKdsh0Wo/hqdefault.jpg)

--- 

Feel free to click on the video titles to watch them!